## Structure gemini outputs in a robust way

In [1]:
from google import genai

# The client gets the API key from the environment variable `GEMINI_API_KEY`.
client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash", contents="""
    List a few asian soups recipies, a yummy description
    list the ingredients. Structure it in json format"""
)
print(response.text)

Here are a few delectable Asian soup recipes, complete with yummy descriptions and ingredients, all structured in JSON format:

```json
[
  {
    "name": "Tom Yum Goong (Spicy Thai Shrimp Soup)",
    "description": "An iconic Thai masterpiece that's an explosion of bold flavors. This hot and sour shrimp soup is incredibly aromatic, zesty, and utterly invigorating, with a perfect balance of spicy chili, sour lime, fragrant lemongrass, and galangal. Each spoonful is a vibrant dance for your taste buds!",
    "ingredients": [
      "Shrimp (peeled and deveined)",
      "Mushrooms (straw or oyster)",
      "Lemongrass (bruised and sliced)",
      "Galangal (sliced)",
      "Kaffir lime leaves (torn)",
      "Bird's eye chilies (crushed or sliced, to taste)",
      "Fish sauce",
      "Lime juice (freshly squeezed)",
      "Tom Yum paste (optional, for convenience)",
      "Cilantro (for garnish)",
      "Chicken or vegetable broth",
      "Evaporated milk or coconut milk (optional, for cre

In [3]:
from pydantic import BaseModel
from google import genai


class Recipe(BaseModel):
    recipe_name: str
    description: str
    ingredients: list[str]


client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="""
    List a few asian soups recipes, a yummy description
    and list the ingredients
    """,
    config={"response_mime_type": "application/json", "response_schema": list[Recipe]},
)

print(response.text)

[{"recipe_name":"Tom Yum Goong","description":"A classic Thai hot and sour soup, bursting with aromatic herbs, tangy lime, and spicy chili, often featuring succulent shrimp. It's an invigorating and complex symphony of flavors that will awaken your taste buds.","ingredients":["Shrimp","Lemongrass","Galangal","Kaffir lime leaves","Lime juice","Fish sauce","Mushrooms","Chili paste","Cherry tomatoes","Cilantro"]},
{"recipe_name":"Ramen (Shoyu)","description":"A deeply savory and comforting Japanese noodle soup with a rich soy-sauce-based broth, springy noodles, and an array of delicious toppings like tender pork, soft-boiled eggs, and crisp nori. Each spoonful is a journey of umami.","ingredients":["Ramen noodles","Pork belly (chashu)","Soy sauce","Chicken or pork broth","Mirin","Sake","Green onions","Soft-boiled egg","Nori (seaweed)","Bok choy"]},
{"recipe_name":"Pho Bo (Vietnamese Beef Noodle Soup)","description":"A fragrant and soul-warming Vietnamese beef noodle soup, characterized by

In [4]:
recipes = response.parsed
type(recipes), len(recipes)

(list, 3)

In [5]:
recipes[0]

Recipe(recipe_name='Tom Yum Goong', description="A classic Thai hot and sour soup, bursting with aromatic herbs, tangy lime, and spicy chili, often featuring succulent shrimp. It's an invigorating and complex symphony of flavors that will awaken your taste buds.", ingredients=['Shrimp', 'Lemongrass', 'Galangal', 'Kaffir lime leaves', 'Lime juice', 'Fish sauce', 'Mushrooms', 'Chili paste', 'Cherry tomatoes', 'Cilantro'])

## Simulate house prices

In [6]:
from typing import Literal

class Home(BaseModel):
    price: int
    monthly_fee: int
    living_area: float
    number_rooms: float
    address: str
    type: Literal["apartment", "house"]

response = client.models.generate_content(
    model="gemini-2.5-flash",   
    contents="""List 50 apartments and houses in Sweden with their monthly_fee, their price, living area, 
    number of rooms, address, type if it is apartment or house. 
    All currencies are in SEK.""",
    config={"response_mime_type": "application/json", "response_schema": list[Home]},
)

homes = response.parsed
homes[:3]

[Home(price=3500000, monthly_fee=2500, living_area=75.5, number_rooms=3.0, address='Västerlånggatan 10, Stockholm', type='apartment'),
 Home(price=6800000, monthly_fee=4500, living_area=120.0, number_rooms=5.0, address='Ekallén 15, Göteborg', type='house'),
 Home(price=2800000, monthly_fee=2800, living_area=60.0, number_rooms=2.0, address='Södra Förstadsgatan 22, Malmö', type='apartment')]

In [7]:
import pandas as pd 

df = pd.DataFrame([dict(home) for home in homes])
df.head()

,price,monthly_fee,living_area,number_rooms,address,type
0,3500000,2500,75.5,3.0,"Västerlånggatan 10, Stockholm",apartment
1,6800000,4500,120.0,5.0,"Ekallén 15, Göteborg",house
2,2800000,2800,60.0,2.0,"Södra Förstadsgatan 22, Malmö",apartment
3,5200000,3800,95.5,4.0,"Björnvägen 7, Uppsala",house
4,4100000,3200,88.0,3.5,"Storgatan 50, Linköping",apartment


In [8]:
df.query("price < 2_000_000")

,price,monthly_fee,living_area,number_rooms,address,type
